In [1]:
import pandas as pd

df = pd.read_csv("../data/oasis_longitudinal.csv")
print(df.shape)
print(df.columns.tolist())
print(df.isna().sum())

df["label"] = (df["CDR"] > 0).astype(int)
print(df["label"].value_counts())
print(df["Subject ID"].nunique())

(373, 15)
['Subject ID', 'MRI ID', 'Group', 'Visit', 'MR Delay', 'M/F', 'Hand', 'Age', 'EDUC', 'SES', 'MMSE', 'CDR', 'eTIV', 'nWBV', 'ASF']
Subject ID     0
MRI ID         0
Group          0
Visit          0
MR Delay       0
M/F            0
Hand           0
Age            0
EDUC           0
SES           19
MMSE           2
CDR            0
eTIV           0
nWBV           0
ASF            0
dtype: int64
label
0    206
1    167
Name: count, dtype: int64
150


In [2]:
import os

print(os.getcwd())
print(os.listdir())

C:\Users\PMLS\Documents\gks-cognitive-xai\notebooks
['.ipynb_checkpoints', '01_data_exploration.ipynb', '02_baseline_and_experiments.ipynb']


In [3]:
# Distribution of original clinical labels
print(df["CDR"].value_counts().sort_index())

# Distribution of groups
print(df["Group"].value_counts())

# Summary statistics
print(df[["Age", "EDUC", "MMSE", "eTIV", "nWBV", "ASF"]].describe())

# Check how many records each subject has
print(df.groupby("Subject ID").size().value_counts().sort_index())

CDR
0.0    206
0.5    123
1.0     41
2.0      3
Name: count, dtype: int64
Group
Nondemented    190
Demented       146
Converted       37
Name: count, dtype: int64
              Age        EDUC        MMSE         eTIV        nWBV         ASF
count  373.000000  373.000000  371.000000   373.000000  373.000000  373.000000
mean    77.013405   14.597855   27.342318  1488.128686    0.729568    1.195461
std      7.640957    2.876339    3.683244   176.139286    0.037135    0.138092
min     60.000000    6.000000    4.000000  1106.000000    0.644000    0.876000
25%     71.000000   12.000000   27.000000  1357.000000    0.700000    1.099000
50%     77.000000   15.000000   29.000000  1470.000000    0.729000    1.194000
75%     82.000000   16.000000   30.000000  1597.000000    0.756000    1.293000
max     98.000000   23.000000   30.000000  2004.000000    0.837000    1.587000
2    94
3    43
4     9
5     4
Name: count, dtype: int64


In [4]:
import warnings
import pandas as pd
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

warnings.filterwarnings("ignore")

df = pd.read_csv("../data/oasis_longitudinal.csv")
df["label"] = (df["CDR"] > 0).astype(int)   # 1 = dementia signs, 0 = none
df["Sex"] = (df["M/F"] == "M").astype(int)  # turn M/F into 1/0

features_B = ["Age", "Sex", "EDUC", "SES", "eTIV", "nWBV"]  # Experiment B: no MMSE
X = df[features_B]
y = df["label"]
groups = df["Subject ID"]   # keeps each person's rows together

cv = GroupKFold(n_splits=5)  # 5 rounds; each person is in the test part once
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

models = {
    # a "dumb" model that always guesses the most common class
    "Baseline (always majority)": DummyClassifier(strategy="most_frequent"),
    # fills missing values and scales the numbers, learned from training data only
    "Logistic Regression": make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(max_iter=1000),
    ),
}

for name, model in models.items():
    res = cross_validate(model, X, y, groups=groups, cv=cv, scoring=scoring)
    print(name)
    for m in scoring:
        s = res["test_" + m]
        print(f"  {m}: {s.mean():.2f} (+/- {s.std():.2f})")

Baseline (always majority)
  accuracy: 0.55 (+/- 0.06)
  precision: 0.00 (+/- 0.00)
  recall: 0.00 (+/- 0.00)
  f1: 0.00 (+/- 0.00)
  roc_auc: 0.50 (+/- 0.00)
Logistic Regression
  accuracy: 0.65 (+/- 0.08)
  precision: 0.63 (+/- 0.15)
  recall: 0.59 (+/- 0.09)
  f1: 0.60 (+/- 0.11)
  roc_auc: 0.73 (+/- 0.10)
